# 🧠 Продвинутый шаблон для соревнований по текстовому пониманию и классификации

Этот ноутбук — **универсальный и расширенный шаблон** для задач:

- текстовой классификации (sentiment, intent, topic, toxicity и т.д.)
- текстового понимания (классификация по содержанию текста)
- мульти-лейбл задач (несколько меток на один текст)
- простой регрессии по тексту (например, score от 0 до 1)

Ключевые фичи:

- Всё настраивается через один блок `CONFIG`
- Поддерживаются **разные форматы данных**: `csv`, `parquet`, `json`, `xlsx`
- Гибкая работа с **любым названием колонок** (через конфиг)
- Модели:
  - константный бейзлайн
  - `TF-IDF + LogisticRegression` (классификация)
  - `TF-IDF + OneVsRest + LogisticRegression` (multilabel)
  - `TF-IDF + Ridge` (регрессия)
  - пример `Transformer` (HuggingFace, для классификации)
- Holdout-валидация и опциональный **K-Fold cross-validation** для TF-IDF моделей
- Авто-обработка сабмишена, в том числе для multilabel/regression

> ⚠️ **Важно:** всё поведение задаётся в блоке `CONFIG` ниже.  
> Открой его, адаптируй под свою задачу и запускай ноутбук по шагам.


In [ ]:
# =============================
# 🔧 CONFIG — ГЛАВНЫЕ НАСТРОЙКИ
# =============================
#
# Здесь вы настраиваете:
# - тип задачи: binary / multiclass / multilabel / regression
# - откуда читать данные и в каком они формате
# - названия колонок (тексты, таргеты, id)
# - тип модели: baseline / tfidf / transformer
# - как валидироваться: holdout / k-fold
#
# Идея: вы меняете ТОЛЬКО этот блок, а весь остальной код подстраивается под конфиг.

from dataclasses import dataclass, field
from typing import List, Optional, Tuple

@dataclass
class DataConfig:
    train_path: str = "data/train.csv"
    test_path: str = "data/test.csv"
    sample_submission_path: Optional[str] = "data/sample_submission.csv"

    text_columns: List[str] = field(default_factory=lambda: ["text"])
    target_columns: List[str] = field(default_factory=lambda: ["label"])
    id_column_candidates: List[str] = field(default_factory=lambda: ["id", "ID", "Id"])

@dataclass
class TaskConfig:
    task_type: str = "multiclass"  # binary / multiclass / multilabel / regression
    labels_are_integers: bool = False
    label_list: Optional[List[str]] = None
    main_metric: str = "f1_macro"

@dataclass
class ModelConfig:
    model_type: str = "tfidf_linear"  # baseline_constant / tfidf_linear / transformer_encoder

    random_seed: int = 42
    use_cross_validation: bool = False
    n_splits: int = 5

    tfidf_max_features: int = 30000
    tfidf_ngram_range: Tuple[int, int] = (1, 2)

    transformer_model_name: str = "distilbert-base-multilingual-cased"
    transformer_max_length: int = 128
    transformer_batch_size: int = 16
    transformer_epochs: int = 2
    transformer_lr: float = 2e-5

@dataclass
class TrainingConfig:
    use_holdout_validation: bool = True
    val_size: float = 0.2

    save_model: bool = True
    model_output_path: str = "models/model.pkl"

@dataclass
class SubmissionConfig:
    submission_path: str = "submissions/submission.csv"
    submission_target_columns: Optional[List[str]] = None


data_config = DataConfig()
task_config = TaskConfig()
model_config = ModelConfig()
training_config = TrainingConfig()
submission_config = SubmissionConfig()

print("✅ CONFIG загружен. При необходимости измените параметры выше и перезапустите ноутбук.")

## 1. Импорты и базовая подготовка окружения

Здесь:
- импортируем библиотеки
- фиксируем сид для воспроизводимости
- готовим функцию загрузки таблиц для разных форматов


In [ ]:
# =============================
# 1. Импорты и фиксирование сидов
# =============================

import os
import random
import numpy as np
import pandas as pd

from typing import Dict, Any

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    mean_squared_error,
    mean_absolute_error,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.multiclass import OneVsRestClassifier
import joblib

try:
    import torch
    from torch.utils.data import Dataset, DataLoader
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
    TRANSFORMERS_AVAILABLE = True
except ImportError:
    print("⚠️ transformers / torch не найдены. "
          "Если хотите использовать transformer_encoder, установите библиотеки:\n"
          "pip install torch transformers")
    TRANSFORMERS_AVAILABLE = False


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except ImportError:
        pass

set_seed(model_config.random_seed)
print(f"✅ Сид зафиксирован: {model_config.random_seed}")


def load_table(path: str) -> pd.DataFrame:
    ext = os.path.splitext(path)[1].lower()
    if ext in [".csv", ""]:
        return pd.read_csv(path)
    elif ext == ".parquet":
        return pd.read_parquet(path)
    elif ext == ".json":
        return pd.read_json(path)
    elif ext in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    else:
        raise ValueError(f"Неизвестное расширение файла: {ext}. "
                         f"Добавьте обработку в функцию load_table().")

## 2. Загрузка данных

- читаем `train` и `test` через `load_table`
- проверяем размеры и первые строки

Если у вас другие названия колонок — просто укажите их в `DataConfig.text_columns`
и `DataConfig.target_columns`.


In [ ]:
# =============================
# 2. Загрузка данных
# =============================

train_path = data_config.train_path
test_path = data_config.test_path

print("Используемые пути:")
print("  train:", train_path)
print("  test :", test_path)

train_df = load_table(train_path)
test_df = load_table(test_path)

print("\nРазмеры датасетов:")
print("  train:", train_df.shape)
print("  test :", test_df.shape)

print("\nПервые строки train:")
display(train_df.head())

print("\nПервые строки test:")
display(test_df.head())

## 3. Подготовка текстов и таргетов

- объединяем несколько текстовых колонок в одну строку
- подготавливаем `y` в зависимости от типа задачи
- при необходимости строим `label2id` и `id2label`


In [ ]:
# =============================
# 3. Подготовка текстов и таргетов
# =============================

def combine_text_columns(df: pd.DataFrame, text_cols):
    if len(text_cols) == 1:
        return df[text_cols[0]].astype(str)
    else:
        return df[text_cols].astype(str).agg(" ".join, axis=1)


text_cols = data_config.text_columns
target_cols = data_config.target_columns

for col in text_cols:
    if col not in train_df.columns:
        raise KeyError(f"Текстовая колонка '{col}' не найдена в train_df.")
    if col not in test_df.columns:
        raise KeyError(f"Текстовая колонка '{col}' не найдена в test_df.")

train_texts = combine_text_columns(train_df, text_cols)
test_texts = combine_text_columns(test_df, text_cols)

print("Пример объединённого текста из train:")
print(train_texts.iloc[0])

task_type = task_config.task_type

for col in target_cols:
    if col not in train_df.columns:
        raise KeyError(f"Таргет-колонка '{col}' не найдена в train_df. Проверьте DataConfig.target_columns.")

if task_type in ["binary", "multiclass"]:
    assert len(target_cols) == 1, "Для binary/multiclass ожидается ОДНА target-колонка"
    raw_labels = train_df[target_cols[0]]

    if task_config.labels_are_integers:
        y = raw_labels.values
        label_list = sorted(np.unique(y))
        label2id = {int(label): int(label) for label in label_list}
        id2label = {int(label): int(label) for label in label_list}
    else:
        if task_config.label_list is not None:
            label_list = task_config.label_list
        else:
            label_list = sorted(raw_labels.unique())
        label2id = {label: i for i, label in enumerate(label_list)}
        id2label = {i: label for label in label_list}
        y = raw_labels.map(label2id).values

    num_labels = len(label_list)
    print(f"Тип задачи: {task_type}, классов: {num_labels}")
    print("label2id:", label2id)

elif task_type == "multilabel":
    y = train_df[target_cols].values
    num_labels = y.shape[1]
    label_list = target_cols
    label2id = {label: i for i, label in enumerate(label_list)}
    id2label = {i: label for label in label_list}
    print(f"Тип задачи: multilabel, классов: {num_labels}")
    print("label_list:", label_list)

elif task_type == "regression":
    y = train_df[target_cols].values
    if y.shape[1] == 1:
        y = y.ravel()
    num_labels = y.shape[1] if y.ndim > 1 else 1
    label_list = target_cols
    label2id = {}
    id2label = {}
    print(f"Тип задачи: regression, размерность таргета: {num_labels}")
else:
    raise ValueError(f"Неизвестный тип задачи: {task_type}")

## 4. Разбиение на train/validation или подготовка кросс-валидации

Здесь формируются:
- `X`, `X_test`
- при необходимости `X_train/X_val` и `y_train/y_val`


In [ ]:
# =============================
# 4. Разбиение данных
# =============================

X = train_texts.values
X_test = test_texts.values

if training_config.use_holdout_validation:
    if task_type in ["binary", "multiclass"]:
        stratify = y
    else:
        stratify = None

    X_train, X_val, y_train, y_val = train_test_split(
        X, y,
        test_size=training_config.val_size,
        random_state=model_config.random_seed,
        stratify=stratify
    )

    print("Используем holdout-валидацию.")
    print("  X_train:", X_train.shape)
    print("  X_val  :", X_val.shape)
else:
    X_train, X_val, y_train, y_val = X, None, y, None
    print("Holdout-валидация отключена. "
          "Можно использовать K-Fold (ModelConfig.use_cross_validation = True).")

## 5. Модели

Реализованы варианты:

1. `baseline_constant` — самый частый класс (binary/multiclass).  
2. `tfidf_linear`:
   - классификация: TF-IDF + LogisticRegression
   - multilabel: TF-IDF + OneVsRest(LogisticRegression)
   - регрессия: TF-IDF + Ridge  
3. `transformer_encoder` — пример трансформера для классификации.


In [ ]:
# =============================
# 5.1. Baseline: constant model
# =============================

class ConstantModel:
    # Всегда предсказывает самый частый класс (binary/multiclass).
    def __init__(self):
        self.constant_class = None

    def fit(self, y):
        values, counts = np.unique(y, return_counts=True)
        self.constant_class = values[np.argmax(counts)]
        print(f"ConstantModel: всегда предсказываем класс {self.constant_class}")

    def predict(self, X):
        return np.full(shape=(len(X),), fill_value=self.constant_class)

In [ ]:
# =============================
# 5.2. TF-IDF-модели
# =============================

class TfidfClassifierModel:
    # TF-IDF + LogisticRegression для binary/multiclass.
    def __init__(self, config: ModelConfig):
        self.config = config
        self.vectorizer = TfidfVectorizer(
            max_features=config.tfidf_max_features,
            ngram_range=config.tfidf_ngram_range
        )
        self.model = LogisticRegression(
            max_iter=1000,
            random_state=config.random_seed,
            n_jobs=-1
        )

    def fit(self, X_train, y_train):
        print("Обучаем TF-IDF (классификация)...")
        X_train_vec = self.vectorizer.fit_transform(X_train)
        print("Векторизовано:", X_train_vec.shape)

        print("Обучаем LogisticRegression...")
        self.model.fit(X_train_vec, y_train)
        print("✅ Обучение завершено.")

    def predict(self, X):
        X_vec = self.vectorizer.transform(X)
        return self.model.predict(X_vec)

    def predict_proba(self, X):
        X_vec = self.vectorizer.transform(X)
        if hasattr(self.model, "predict_proba"):
            return self.model.predict_proba(X_vec)
        else:
            return None


class TfidfMultiLabelModel:
    # TF-IDF + OneVsRest(LogisticRegression) для multilabel.
    def __init__(self, config: ModelConfig):
        self.config = config
        self.vectorizer = TfidfVectorizer(
            max_features=config.tfidf_max_features,
            ngram_range=config.tfidf_ngram_range
        )
        base_clf = LogisticRegression(
            max_iter=1000,
            random_state=config.random_seed,
            n_jobs=-1
        )
        self.model = OneVsRestClassifier(base_clf)

    def fit(self, X_train, y_train):
        print("Обучаем TF-IDF (multilabel)...")
        X_train_vec = self.vectorizer.fit_transform(X_train)
        print("Векторизовано:", X_train_vec.shape)

        print("Обучаем OneVsRest(LogisticRegression)...")
        self.model.fit(X_train_vec, y_train)
        print("✅ Обучение завершено.")

    def predict(self, X):
        X_vec = self.vectorizer.transform(X)
        return self.model.predict(X_vec)


class TfidfRegressorModel:
    # TF-IDF + Ridge для регрессии.
    def __init__(self, config: ModelConfig):
        self.config = config
        self.vectorizer = TfidfVectorizer(
            max_features=config.tfidf_max_features,
            ngram_range=config.tfidf_ngram_range
        )
        self.model = Ridge(random_state=config.random_seed)

    def fit(self, X_train, y_train):
        print("Обучаем TF-IDF (регрессия)...")
        X_train_vec = self.vectorizer.fit_transform(X_train)
        print("Векторизовано:", X_train_vec.shape)

        print("Обучаем Ridge...")
        self.model.fit(X_train_vec, y_train)
        print("✅ Обучение завершено.")

    def predict(self, X):
        X_vec = self.vectorizer.transform(X)
        return self.model.predict(X_vec)

In [ ]:
# =============================
# 5.3. Transformer encoder (пример)
# =============================

if TRANSFORMERS_AVAILABLE:

    class TextDataset(Dataset):
        def __init__(self, texts, labels=None, tokenizer=None, max_length=128):
            self.texts = texts
            self.labels = labels
            self.tokenizer = tokenizer
            self.max_length = max_length

        def __len__(self):
            return len(self.texts)

        def __getitem__(self, idx):
            text = str(self.texts[idx])
            enc = self.tokenizer(
                text,
                truncation=True,
                padding="max_length",
                max_length=self.max_length,
                return_tensors="pt"
            )
            item = {k: v.squeeze(0) for k, v in enc.items()}
            if self.labels is not None:
                item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
            return item


    class TransformerModel:
        def __init__(self, model_config: ModelConfig, num_labels: int):
            self.config = model_config
            self.num_labels = num_labels

            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            print("Используем устройство:", self.device)

            self.tokenizer = AutoTokenizer.from_pretrained(self.config.transformer_model_name)
            self.model = AutoModelForSequenceClassification.from_pretrained(
                self.config.transformer_model_name,
                num_labels=self.num_labels
            )
            self.model.to(self.device)

        def fit(self, X_train, y_train, X_val=None, y_val=None):
            train_dataset = TextDataset(
                texts=X_train,
                labels=y_train,
                tokenizer=self.tokenizer,
                max_length=self.config.transformer_max_length
            )

            train_loader = DataLoader(
                train_dataset,
                batch_size=self.config.transformer_batch_size,
                shuffle=True
            )

            optimizer = torch.optim.AdamW(self.model.parameters(), lr=self.config.transformer_lr)
            total_steps = len(train_loader) * self.config.transformer_epochs
            scheduler = get_linear_schedule_with_warmup(
                optimizer,
                num_warmup_steps=int(0.1 * total_steps),
                num_training_steps=total_steps
            )

            self.model.train()
            for epoch in range(self.config.transformer_epochs):
                print(f"Эпоха {epoch+1}/{self.config.transformer_epochs}")
                epoch_loss = 0.0
                for batch in train_loader:
                    batch = {k: v.to(self.device) for k, v in batch.items()}

                    optimizer.zero_grad()
                    outputs = self.model(**batch)
                    loss = outputs.loss
                    loss.backward()

                    optimizer.step()
                    scheduler.step()

                    epoch_loss += loss.item()

                print(f"Средний loss за эпоху: {epoch_loss / len(train_loader):.4f}")

        def _predict_logits(self, X):
            self.model.eval()
            dataset = TextDataset(
                texts=X,
                labels=None,
                tokenizer=self.tokenizer,
                max_length=self.config.transformer_max_length
            )
            loader = DataLoader(dataset, batch_size=self.config.transformer_batch_size, shuffle=False)
            all_logits = []
            with torch.no_grad():
                for batch in loader:
                    batch = {k: v.to(self.device) for k, v in batch.items()}
                    outputs = self.model(**batch)
                    logits = outputs.logits
                    all_logits.append(logits.cpu().numpy())
            all_logits = np.concatenate(all_logits, axis=0)
            return all_logits

        def predict(self, X):
            logits = self._predict_logits(X)
            preds = logits.argmax(axis=1)
            return preds

        def predict_proba(self, X):
            logits = self._predict_logits(X)
            probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
            return probs

else:
    TransformerModel = None

## 6. Обучение выбранной модели и оценка качества

Здесь:
- создаём модель по конфигу
- обучаем (holdout или k-fold)
- считаем метрики


In [ ]:
# =============================
# 6. Обучение и валидация
# =============================

def get_model(model_config: ModelConfig, task_type: str, num_labels: int):
    if model_config.model_type == "baseline_constant":
        assert task_type in ["binary", "multiclass"], \
            "baseline_constant поддерживает только binary/multiclass"
        return ConstantModel()

    elif model_config.model_type == "tfidf_linear":
        if task_type in ["binary", "multiclass"]:
            return TfidfClassifierModel(model_config)
        elif task_type == "multilabel":
            return TfidfMultiLabelModel(model_config)
        elif task_type == "regression":
            return TfidfRegressorModel(model_config)
        else:
            raise ValueError(f"tfidf_linear не знает тип задачи: {task_type}")

    elif model_config.model_type == "transformer_encoder":
        assert TRANSFORMERS_AVAILABLE, "Transformers не установлены!"
        assert task_type in ["binary", "multiclass"], \
            "transformer_encoder поддерживает только binary/multiclass"
        return TransformerModel(model_config, num_labels=num_labels)

    else:
        raise ValueError(f"Неизвестный тип модели: {model_config.model_type}")


def evaluate_classification(y_true, y_pred, label_list=None):
    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average="macro")
    f1_weighted = f1_score(y_true, y_pred, average="weighted")

    print("\n📊 Результаты на валидации:")
    print(f"Accuracy   : {acc:.4f}")
    print(f"F1-macro   : {f1_macro:.4f}")
    print(f"F1-weighted: {f1_weighted:.4f}")

    print("\nОтчёт classification_report:")
    if label_list is not None:
        print(classification_report(y_true, y_pred, target_names=[str(x) for x in label_list]))
    else:
        print(classification_report(y_true, y_pred))


def evaluate_multilabel(y_true, y_pred):
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    f1_micro = f1_score(y_true, y_pred, average="micro", zero_division=0)

    print("\n📊 Результаты на валидации (multilabel):")
    print(f"F1-macro: {f1_macro:.4f}")
    print(f"F1-micro: {f1_micro:.4f}")


def evaluate_regression(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = mse ** 0.5
    mae = mean_absolute_error(y_true, y_pred)

    print("\n📊 Результаты на валидации (regression):")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE : {mae:.4f}")


model = get_model(model_config, task_type, num_labels)

if training_config.use_holdout_validation and X_val is not None:
    print("=== Обучение с holdout-валидацией ===")
    if model_config.model_type == "baseline_constant":
        model.fit(y_train)
    elif model_config.model_type == "transformer_encoder":
        model.fit(X_train, y_train, X_val, y_val)
    else:
        model.fit(X_train, y_train)

    if task_type in ["binary", "multiclass"]:
        y_val_pred = model.predict(X_val)
        evaluate_classification(y_val, y_val_pred, label_list)

    elif task_type == "multilabel":
        y_val_pred = model.predict(X_val)
        evaluate_multilabel(y_val, y_val_pred)

    elif task_type == "regression":
        y_val_pred = model.predict(X_val)
        evaluate_regression(y_val, y_val_pred)

elif (not training_config.use_holdout_validation
      and model_config.use_cross_validation
      and model_config.model_type == "tfidf_linear"
      and task_type in ["binary", "multiclass"]):
    print("=== Обучение с K-Fold cross-validation (TF-IDF классификация) ===")

    skf = StratifiedKFold(
        n_splits=model_config.n_splits,
        shuffle=True,
        random_state=model_config.random_seed
    )

    fold_metrics = []
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n--- Fold {fold+1}/{model_config.n_splits} ---")
        X_tr, X_va = X[train_idx], X[val_idx]
        y_tr, y_va = y[train_idx], y[val_idx]

        fold_model = TfidfClassifierModel(model_config)
        fold_model.fit(X_tr, y_tr)
        y_va_pred = fold_model.predict(X_va)

        acc = accuracy_score(y_va, y_va_pred)
        f1_macro = f1_score(y_va, y_va_pred, average="macro")
        print(f"Fold {fold+1} — ACC: {acc:.4f}, F1-macro: {f1_macro:.4f}")
        fold_metrics.append((acc, f1_macro))

    fold_metrics = np.array(fold_metrics)
    print("\n=== Средние метрики по фолдам ===")
    print(f"ACC mean: {fold_metrics[:,0].mean():.4f} ± {fold_metrics[:,0].std():.4f}")
    print(f"F1m mean: {fold_metrics[:,1].mean():.4f} ± {fold_metrics[:,1].std():.4f}")

    print("\nОбучаем финальную модель на всех данных train...")
    model = TfidfClassifierModel(model_config)
    model.fit(X, y)

else:
    print("=== Обучение на всех данных train без валидации ===")
    if model_config.model_type == "baseline_constant":
        model.fit(y)
    elif model_config.model_type == "transformer_encoder":
        model.fit(X, y)
    else:
        model.fit(X, y)


if training_config.save_model and model_config.model_type in ["baseline_constant", "tfidf_linear"]:
    os.makedirs(os.path.dirname(training_config.model_output_path), exist_ok=True)
    joblib.dump(model, training_config.model_output_path)
    print(f"\n💾 Модель сохранена в {training_config.model_output_path}")

## 7. Предсказания на тесте и формирование сабмишена

- считаем предсказания на `test`
- приводим формат под тип задачи
- сохраняем `submission.csv`


In [ ]:
# =============================
# 7. Сабмишен
# =============================

print("Делаем предсказания на test...")
if model_config.model_type == "baseline_constant":
    model.fit(y)
    test_pred = model.predict(X_test)
else:
    test_pred = model.predict(X_test)

test_pred = np.array(test_pred)

if task_type in ["binary", "multiclass"]:
    if not task_config.labels_are_integers:
        test_output = np.array([id2label[int(i)] for i in test_pred])
    else:
        test_output = test_pred

elif task_type in ["multilabel", "regression"]:
    if test_pred.ndim == 1:
        test_output = test_pred.reshape(-1, 1)
    else:
        test_output = test_pred
else:
    raise ValueError(f"Неизвестный тип задачи: {task_type}")


submission_path = submission_config.submission_path
os.makedirs(os.path.dirname(submission_path), exist_ok=True)

if data_config.sample_submission_path and os.path.exists(data_config.sample_submission_path):
    sample_sub = load_table(data_config.sample_submission_path)
    print("Загружен sample_submission:", data_config.sample_submission_path)
else:
    sample_sub = pd.DataFrame()
    id_col = None
    for c in data_config.id_column_candidates:
        if c in test_df.columns:
            id_col = c
            break
    if id_col is None:
        sample_sub["id"] = np.arange(len(test_df))
        id_col = "id"
    else:
        sample_sub[id_col] = test_df[id_col]

out_cols = submission_config.submission_target_columns or data_config.target_columns

if task_type in ["binary", "multiclass"]:
    if len(out_cols) != 1:
        raise ValueError("Для binary/multiclass ожидается ОДНА колонка таргета в сабмишене.")
    col = out_cols[0]
    sample_sub[col] = test_output

elif task_type in ["multilabel", "regression"]:
    if test_output.ndim == 1:
        test_output_2d = test_output.reshape(-1, 1)
    else:
        test_output_2d = test_output

    if len(out_cols) != test_output_2d.shape[1]:
        raise ValueError(
            f"Количество target-колонок ({len(out_cols)}) не совпадает с числом предсказаний "
            f"({test_output_2d.shape[1]}). Проверьте SubmissionConfig.submission_target_columns."
        )
    for i, col in enumerate(out_cols):
        sample_sub[col] = test_output_2d[:, i]

sample_sub.to_csv(submission_path, index=False)
print(f"✅ Сабмишен сохранён в {submission_path}")
display(sample_sub.head())

## 8. Идеи для расширения шаблона

### Multilabel
- Порог по вероятностям, если понадобится (`predict_proba` + threshold)
- Сложные метрики (per-class F1, PR-AUC и т.д.)

### Регрессия
- Другие модели: SVR, RandomForestRegressor, GradientBoosting и т.п.
- Лог-трансформации таргета, нормализация

### Трансформеры
- Поддержка multilabel через BCEWithLogitsLoss
- Поддержка регрессии через MSELoss
- K-fold, усреднение логитов, fp16

### Обработка текста
- Очистка HTML, эмодзи, ссылок
- Лемматизация / стемминг под язык
- Добавление простых фич (длина текста, количество !, ? и т.д.)

### Эксперименты и логирование
- Ведение таблички с результатами (конфиг + метрики)
- Использование `wandb`, `mlflow` и т.п.

---

Это **универсальный стартовый шаблон**. Под конкретную задачу вы можете:
- вырезать лишнее,
- дописать кастомную предобработку,
- добавить более сложные модели и метрики.
